# DualSentinel — Smoke Test (LMD-2023, max 20 LLM calls)

Equivalente a:

```bash
python src/pipeline.py \
  --input "data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv" \
  --dataset lmd \
  --evaluate \
  --max-llm-calls 20
```

**Objetivo:** validar end-to-end o pipeline (parse → windowing → baseline → tagger+KB → heuristic scorer → SLM → Judge → report) num dataset grande, mas limitando a **20 chamadas ao Ollama** em cada estágio LLM (SLM Analyst e LLM Judge) para obter resultados rápidos.

> Pré-requisitos: `ollama serve` a correr, modelos `phi3:medium` e `llama3.1` já puxados, env conda/venv com `requirements.txt` instalado.


In [ ]:
"""Setup: caminhos, imports e auto-reload de módulos do src/."""
import sys, os
from pathlib import Path

# Garantir que o src/ do DualSentinel está no sys.path
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Trabalhar a partir da raiz do DualSentinel para os caminhos relativos baterem
os.chdir(PROJECT_ROOT)

# Auto-reload: pickup imediato de edits em src/*.py sem restart do kernel
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

print("Working dir:", Path.cwd())
print("src on path:", SRC_DIR.exists())


Working dir: d:\ISEP\Challange-3\DualSentinel
src on path: True


## 1. Parâmetros do smoke test

Mantém em sintonia com a CLI. Ajusta `INPUT_PATH` se o ficheiro estiver noutro sítio.


In [2]:
from datetime import datetime

INPUT_PATH    = Path("data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv")
DATASET       = "lmd"
EVALUATE      = True
MAX_LLM_CALLS = 20         # cap por estágio (SLM e Judge)
THRESHOLD     = None       # None → usa ANOMALY_THRESHOLD do .env (default 0.6)
USE_KB        = True
SKIP_JUDGE    = False

ts = datetime.now().strftime("%Y-%m-%d_%H-%M")
OUTPUT_DIR = Path("results") / f"smoke_max{MAX_LLM_CALLS}_{ts}"

assert INPUT_PATH.exists(), f"Input não encontrado: {INPUT_PATH.resolve()}"
print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_DIR}")
print(f"Cap    : {MAX_LLM_CALLS} chamadas Ollama por estágio (SLM + Judge)")


Input  : data\samples\LMD-2023 [1.75M Elements - Normal]checked.csv
Output : results\smoke_max20_2026-04-19_23-41
Cap    : 20 chamadas Ollama por estágio (SLM + Judge)


## 2. Sanity check — Ollama disponível


In [3]:
import urllib.request, json as _json
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
        tags = _json.loads(r.read())
    models = [m["name"] for m in tags.get("models", [])]
    print("Ollama OK. Modelos disponíveis:")
    for m in models:
        print(" •", m)
    needed = [os.getenv("SLM_MODEL", "phi3:medium"), os.getenv("JUDGE_MODEL", "llama3.1")]
    missing = [m for m in needed if not any(m.split(":")[0] in x for x in models)]
    if missing:
        print(f"\n⚠️  Em falta: {missing}. Corre: ollama pull {' && ollama pull '.join(missing)}")
except Exception as e:
    print(f"⚠️  Ollama não acessível em :11434 ({e}). Arranca com `ollama serve`.")


Ollama OK. Modelos disponíveis:
 • phi4:14b
 • phi3:medium
 • qwen3:8b
 • phi3:mini
 • llama3.2:latest

⚠️  Em falta: ['llama3.1']. Corre: ollama pull llama3.1


## 3. Correr o pipeline

Chama diretamente `run_pipeline(...)` (mesma função usada pela CLI).


In [ ]:
import time, logging, importlib
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

# Force-reload em cascata: garante que edits a src/schema.py, src/preprocessor.py,
# src/detectors.py, etc. são apanhados mesmo sem restart do kernel.
for mod_name in [
    "schema", "embeddings", "chains", "attack_kb",
    "detectors", "utils", "preprocessor",
    "slm_analyst", "llm_judge", "pipeline",
]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from pipeline import run_pipeline

t0 = time.time()
result = run_pipeline(
    input_path    = INPUT_PATH,
    dataset       = DATASET,
    output_dir    = OUTPUT_DIR,
    skip_judge    = SKIP_JUDGE,
    threshold     = THRESHOLD,
    evaluate      = EVALUATE,
    use_kb        = USE_KB,
    max_llm_calls = MAX_LLM_CALLS,
)
elapsed = time.time() - t0
print(f"\n✓ Pipeline concluído em {elapsed/60:.1f} min")
print(f"   Output dir: {result.get('output_dir')}")
print(f"   Report    : {result.get('report')}")


Step 1: Parsing LMD-2023 [1.75M Elements - Normal]checked.csv...

TypeError: arg must be a list, tuple, 1-d array, or Series

## 4. Distribuição do `detector_score` (1ª sentinela)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open(OUTPUT_DIR / "windows_scored.json", encoding="utf-8") as f:
    windows = json.load(f)

scores = pd.Series([w["detector_score"] for w in windows], name="detector_score")
eff_thr = THRESHOLD if THRESHOLD is not None else float(os.getenv("ANOMALY_THRESHOLD", 0.6))

print(f"Total janelas       : {len(scores)}")
print(f"Threshold efetivo   : {eff_thr}")
print(f"Acima do threshold  : {(scores >= eff_thr).sum()} ({(scores >= eff_thr).mean()*100:.2f}%)")
print(scores.describe())

fig, ax = plt.subplots(figsize=(8, 3))
scores.hist(bins=40, ax=ax, edgecolor="black")
ax.axvline(eff_thr, color="red", linestyle="--", label=f"threshold={eff_thr}")
ax.set_xlabel("detector_score"); ax.set_ylabel("janelas"); ax.set_title("Heuristic scorer — distribuição")
ax.legend(); plt.tight_layout(); plt.show()


## 5. ATT&CK hits — rule tagger vs KB


In [ ]:
from collections import Counter

rule_tids, kb_tids = Counter(), Counter()
for w in windows:
    for h in w.get("attck_hits", []):
        tid = h.get("technique") or h.get("technique_id") or "?"
        (rule_tids if h.get("source") == "rule" else kb_tids)[tid] += 1

top_rules = pd.DataFrame(rule_tids.most_common(10), columns=["technique", "count_rule"])
top_kb    = pd.DataFrame(kb_tids.most_common(10),   columns=["technique", "count_kb"])
print("Top regras determinísticas:"); print(top_rules.to_string(index=False))
print("\nTop candidatos KB:");          print(top_kb.to_string(index=False))


## 6. SLM Analyst (Phi-3) — pré-diagnósticos


In [ ]:
slm_path = OUTPUT_DIR / "slm_analyses.json"
if slm_path.exists():
    slm = json.loads(slm_path.read_text(encoding="utf-8"))
    df_slm = pd.DataFrame(slm)
    print(f"Total análises SLM: {len(df_slm)}")
    if "risk_level" in df_slm:
        print("\nRisk level:");           print(df_slm["risk_level"].value_counts())
    if "needs_deep_analysis" in df_slm:
        print("\nEscalado p/ Judge:");    print(df_slm["needs_deep_analysis"].value_counts())
    df_slm[["window_start", "pre_score", "risk_level", "summary"]].head(10)
else:
    print("Sem slm_analyses.json (SLM ignorado).")


## 7. LLM Judge (Llama 3.1) — veredito final


In [ ]:
judge_path = OUTPUT_DIR / "judge_results.json"
if judge_path.exists():
    judge = json.loads(judge_path.read_text(encoding="utf-8"))
    df_j = pd.DataFrame(judge)
    print(f"Total julgamentos: {len(df_j)}  (cap esperado ≤ {MAX_LLM_CALLS})")
    print("\nVeredito:");           print(df_j["verdict"].value_counts())
    print("\nFP risk:");             print(df_j["fp_risk"].value_counts())
    print(f"\nAnomaly score: média={df_j['anomaly_score'].mean():.2f}  máx={df_j['anomaly_score'].max()}")

    cols = ["window_start", "anomaly_score", "verdict", "detector_score", "fp_risk", "rationale"]
    df_top = df_j.sort_values("anomaly_score", ascending=False).head(10)[cols]
    df_top["rationale"] = df_top["rationale"].str.slice(0, 140) + "…"
    df_top
else:
    print("Sem judge_results.json (Judge ignorado).")


## 8. Inspecionar a janela mais suspeita (evidence pack + veredito)


In [ ]:
ep_path = OUTPUT_DIR / "evidence_packs.json"
if ep_path.exists() and judge_path.exists():
    eps = {e["window_start"]: e for e in json.loads(ep_path.read_text(encoding="utf-8"))}
    top = max(judge, key=lambda r: r.get("anomaly_score", 0))
    print(f"=== Janela {top['window_start']} → {top['window_end']} ===")
    print(f"Veredito : {top['verdict']}  |  score {top['anomaly_score']}/10  |  FP risk {top['fp_risk']}")
    print(f"Detector : {top['detector_score']:.3f}")
    print(f"\nRationale:\n  {top['rationale']}")
    if top.get("techniques"):
        print("\nTécnicas:")
        for t in top["techniques"]:
            print(f"  • {t.get('technique_id')} {t.get('name')} (conf {t.get('confidence')}) — {t.get('evidence', '')[:100]}")
    if top.get("unsupported_claims"):
        print(f"\nUnsupported claims: {top['unsupported_claims']}")

    ep = eps.get(top["window_start"])
    if ep:
        print("\n--- Evidence pack (excerto) ---")
        print(ep["evidence_pack"][:2500])
        if len(ep["evidence_pack"]) > 2500:
            print(f"... [+{len(ep['evidence_pack']) - 2500} chars]")
else:
    print("Evidence packs ou judge_results em falta.")


## 9. Relatório Markdown gerado


In [ ]:
from IPython.display import Markdown, display
report = Path(result["report"])
print(f"Relatório: {report}")
display(Markdown(report.read_text(encoding="utf-8")))
